In [9]:
import re

# import spacy
import pandas as pd
import torch
from transformers import pipeline
from sklearn.cluster import KMeans

from umap import UMAP
from hdbscan import HDBSCAN
from sentence_transformers import SentenceTransformer

import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords



from sklearn.feature_extraction.text import CountVectorizer

from keybert import KeyBERT

from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired, TextGeneration
from bertopic.vectorizers import ClassTfidfTransformer  
from bertopic.dimensionality import BaseDimensionalityReduction

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\vallo\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [10]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
stopWords = stopwords.words('russian')

news = pd.read_csv("../data/raw/news.csv")


In [11]:
news.dropna(subset=['text'], inplace=True)

def clean_date(date_str):
    try:
        # Используем регулярное выражение для извлечения даты в формате YYYY-MM-DD
        match = re.search(r'(\d{4}-\d{2}-\d{2})', str(date_str))
        if match:
            return match.group(1)
        return date_str
    except:
        return date_str

news['date'] = news['date'].apply(clean_date)
news.shape

(1814, 2)

# Text classification

In [12]:
embedding_model = SentenceTransformer("deepvk/USER2-base", device=device)
20

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/359 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/14.9k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.27k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/21.2k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.75M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/837 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

20

# Topic modeling

In [22]:
embedding_model_name = "deepvk-USER2-base"
representation_model_name = 'rut5-base-multitask'

In [13]:
dim_model = UMAP(n_neighbors=15, n_components=10, min_dist=0.0, metric='cosine')
# cluster_model = HDBSCAN(min_cluster_size=15, metric='euclidean', cluster_selection_method='eom', prediction_data=True)
cluster_model = KMeans(n_clusters=10, random_state=42)
vectorizer_model = CountVectorizer(stop_words=stopWords)
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)

In [14]:

representation_model = KeyBERTInspired()

# generator = pipeline('text2text-generation', model='cointegrated/rut5-base-multitask', device=device, batch_size=100)
# representation_model = TextGeneration(generator)

In [15]:
topic_model = BERTopic(
  language="russian",
  embedding_model=embedding_model,          # Step 1 - Extract embeddings
  umap_model=dim_model,                     # Step 2 - Reduce dimensionality
  hdbscan_model=cluster_model,              # Step 3 - Cluster reduced embeddings
  vectorizer_model=vectorizer_model,        # Step 4 - Tokenize topics
  ctfidf_model=ctfidf_model,                # Step 5 - Extract topic words
  representation_model=representation_model # Step 6 - (Optional) Fine-tune topic represenations
)

In [16]:
# Скрываем предупреждения
import warnings
warnings.filterwarnings('ignore')


topics, probs = topic_model.fit_transform(news['text'])

In [17]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,0,401,0_сообщалось_года_обнаружили_задержали,"[сообщалось, года, обнаружили, задержали, чело...",[Россияне на всю жизнь оставили своего ребенка...
1,1,298,1_года_российских_обновление_связи,"[года, российских, обновление, связи, оператор...","[По итогам 2019 года, по сравнению с данными и..."
2,2,247,2_россиян_нацпроекта_службе_нацпроект,"[россиян, нацпроекта, службе, нацпроект, росси...","[Тамбовская, Белгородская области и Республика..."
3,3,189,3_нафтогаза_газопровода_газопровод_контракт,"[нафтогаза, газопровода, газопровод, контракт,...",[Министр иностранных дел РФ Сергей Лавров и го...
4,4,164,4_бизнес_forbes_бизнесменов_forbes_young,"[бизнес, forbes, бизнесменов, forbes_young, fo...",[Мировой опыт от обладателей Каннских львов и ...
5,5,152,5_reuters_бирже_турции_bloomberg,"[reuters, бирже, турции, bloomberg, американск...",[Индекс Мосбиржи — индикатор наиболее ликвидн...
6,6,108,6_шеремета_подрыва_шереметом_кузьменко,"[шеремета, подрыва, шереметом, кузьменко, расс...",[Материалы по делу об убийстве журналиста Павл...
7,7,102,7_переговорах_переговоров_донецкой_владимир,"[переговорах, переговоров, донецкой, владимир,...","[Президент Украины Владимир Зеленский считает,..."
8,8,77,8_спорт_олимпийского_олимпийских_олимпийские,"[спорт, олимпийского, олимпийских, олимпийские...",[Глава Федерации лыжных гонок России (ФЛГР) Ел...
9,9,76,9_пожар_кузнецов_крейсер_возгорание,"[пожар, кузнецов, крейсер, возгорание, пожара,...",[Ситуация с пожаром на крейсере Адмирал Кузнец...


In [18]:
ngram_range = "3x3"

In [19]:
# Создаем более осмысленные названия для топиков
# Используем KeyBERT для генерации ключевых фраз из документов каждого топика



# Инициализируем модель KeyBERT
keybert_model = KeyBERT(model=embedding_model)

# Получаем информацию о топиках
topic_info = topic_model.get_topic_info()
topic_docs = {}

# Для каждого топика (кроме -1, который означает выбросы) получаем репрезентативные документы
for topic_id in topic_info[topic_info['Topic'] != -1]['Topic']:
    # Получаем документы для данного топика
    documents = topic_model.get_representative_docs(topic_id)
    topic_docs[topic_id] = ' '.join(documents)

# Создаем словарь для хранения новых названий топиков
topic_names = {}

# Для каждого топика генерируем ключевые фразы
for topic_id, doc in topic_docs.items():
    # Извлекаем ключевые фразы (2 слова) из документов топика
    keywords = keybert_model.extract_keywords(doc, keyphrase_ngram_range=(3, 3), stop_words=stopWords, top_n=1)
    
    if keywords:
        # Берем первую ключевую фразу как название топика
        topic_names[topic_id] = keywords[0][0]
    else:
        # Если не удалось извлечь фразу, используем оригинальное название
        words = topic_model.get_topic(topic_id)
        topic_names[topic_id] = f"Топик_{topic_id}_{words[0][0]}_{words[1][0]}"

# Переименовываем топики в модели
topic_model.set_topic_labels(topic_names)

# Выводим обновленную информацию о топиках
# print("Топики с новыми названиями:")
# display(topic_model.get_topic_info()[1:11])


In [20]:
topic_model.get_topic_info()

,Topic,Count,Name,CustomName,Representation,Representative_Docs
0,0,401,0_сообщалось_года_обнаружили_задержали,хангошвили убит выстрелом,"[сообщалось, года, обнаружили, задержали, чело...",[Россияне на всю жизнь оставили своего ребенка...
1,1,298,1_года_российских_обновление_связи,пиратское вещание телеканалов,"[года, российских, обновление, связи, оператор...","[По итогам 2019 года, по сравнению с данными и..."
2,2,247,2_россиян_нацпроекта_службе_нацпроект,реализован нацпроект экология,"[россиян, нацпроекта, службе, нацпроект, росси...","[Тамбовская, Белгородская области и Республика..."
3,3,189,3_нафтогаза_газопровода_газопровод_контракт,газопровод северный поток,"[нафтогаза, газопровода, газопровод, контракт,...",[Министр иностранных дел РФ Сергей Лавров и го...
4,4,164,4_бизнес_forbes_бизнесменов_forbes_young,forbes_business forbesliferussia forbeswomanru...,"[бизнес, forbes, бизнесменов, forbes_young, fo...",[Мировой опыт от обладателей Каннских львов и ...
5,5,152,5_reuters_бирже_турции_bloomberg,апреля котировки опустились,"[reuters, бирже, турции, bloomberg, американск...",[Индекс Мосбиржи — индикатор наиболее ликвидн...
6,6,108,6_шеремета_подрыва_шереметом_кузьменко,подозреваемых убийстве шеремета,"[шеремета, подрыва, шереметом, кузьменко, расс...",[Материалы по делу об убийстве журналиста Павл...
7,7,102,7_переговорах_переговоров_донецкой_владимир,переговоров нормандской четверки,"[переговорах, переговоров, донецкой, владимир,...","[Президент Украины Владимир Зеленский считает,..."
8,8,77,8_спорт_олимпийского_олимпийских_олимпийские,претензий российскому олимпийскому,"[спорт, олимпийского, олимпийских, олимпийские...",[Глава Федерации лыжных гонок России (ФЛГР) Ел...
9,9,76,9_пожар_кузнецов_крейсер_возгорание,пострадавших возгорании крейсере,"[пожар, кузнецов, крейсер, возгорание, пожара,...",[Ситуация с пожаром на крейсере Адмирал Кузнец...


In [23]:
topic_model.get_topic_info().to_csv(f'../data/interim/topics-{representation_model_name}-{embedding_model_name}-{ngram_range}-Kmeans10.csv', index=False)

In [276]:
topic_model.visualize_topics()

## Pipeline with translation

In [27]:
# Создаем датафрейм для работы с моделью
df = news[['text']].copy()
translator = pipeline("translation", model="Helsinki-NLP/opus-mt-ru-en", device=device)

# Функция для перевода текста с русского на английский
def translate_text(text: list[str]) -> list[str]:
        
    
    # Ограничиваем длину текста для перевода (модель имеет ограничения)


    max_length = 512
    if len(text) > max_length:
        text = text[:max_length]
    # Выполняем перевод
    result = translator(" ".join(text))
    
    # Возвращаем переведенный текст
    return result[0]['translation_text'].split(" ")

# Применяем функцию перевода к представлениям тем

translated_representations = []

for topic in topic_representations:
    # Получаем текущее представление темы
    # Переводим слова темы
    translated_words = translate_text(topic)
    # Обновляем название темы с переводом
    translated_representations.append(translated_words)


Device set to use cuda
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


---

## Отрисовка кластеров

In [11]:
# Получаем эмбеддинги документов
embeddings = embedding_model.encode(news['text'].tolist(), show_progress_bar=True)
embeddings.shape  # Выводим размерность полученных эмбеддингов

dim_2d_model = UMAP(n_neighbors=15, n_components=2, min_dist=0.0, metric='cosine')
zipped_embs = dim_2d_model.fit_transform(embeddings)
cluster_model.fit(zipped_embs)


Batches:   0%|          | 0/57 [00:00<?, ?it/s]

HDBSCAN(min_cluster_size=15, prediction_data=True)

In [12]:
zipped_embs.shape

(1814, 2)

In [168]:
import pandas as pd

data = pd.read_csv('../data/interim/topics-KeyBERTInspired-deepvk-USER2-base-5x5.csv')

In [171]:
data.head()

,Topic,Count,Name,CustomName,Representation,Representative_Docs
0,-1,404,-1_кузьменко_журналиста_антоненко_подозреваемых,-1_кузьменко_журналиста_антоненко_подозреваемых,"['кузьменко', 'журналиста', 'антоненко', 'подо...",['Материалы по делу об убийстве журналиста Пав...
1,0,284,0_сообщалось_обнаружили_года_ребенка,видео посвященное годовщине свадьбы кеосаяном,"['сообщалось', 'обнаружили', 'года', 'ребенка'...","['В Санкт-Петербурге задержали группу парней, ..."
2,1,169,1_reuters_торги_бирже_мосбиржа,лукойл сообщил падении прибыли 27,"['reuters', 'торги', 'бирже', 'мосбиржа', 'инв...",['«Лукойл» сообщил о падении прибыли на 27%: ч...
3,2,124,2_россиян_нацпроекта_российских_получат,которого российских образовательных учреждения...,"['россиян', 'нацпроекта', 'российских', 'получ...",['Росгосцирк предложил Министерству Культуры Р...
4,3,84,3_донецкой_переговорах_переговоров_владимир,проведения встречи нормандском формате украинс...,"['донецкой', 'переговорах', 'переговоров', 'вл...",['На следующей встрече лидеров стран «нормандс...


In [172]:
data['Representation'].tolist()

["['кузьменко', 'журналиста', 'антоненко', 'подозреваемых', 'года', 'левченко', 'шеремета', 'суд', 'российские', 'эксперты']",
 "['сообщалось', 'обнаружили', 'года', 'ребенка', 'инцидент', 'умер', 'ребенок', '2018', 'известно', 'стал']",
 "['reuters', 'торги', 'бирже', 'мосбиржа', 'инвесторам', 'доходность', 'bloomberg', 'акционеры', 'мосбиржи', 'инвесторов']",
 "['россиян', 'нацпроекта', 'российских', 'получат', 'нацпроект', 'региона', 'отметил', 'подмосковье', 'года', 'программы']",
 "['донецкой', 'переговорах', 'переговоров', 'владимир', 'зеленского', 'владимира', 'украина', 'нормандском', 'соглашений', 'положений']",
 "['спорт', 'олимпийского', 'олимпийская', 'олимпийских', 'олимпийские', 'антидопинговое', 'олимпийской', 'спортсменов', 'российское', 'олимпиаде']",
 "['песков', 'чиновников', 'оскорбление', 'закон', 'приговорили', 'госдуму', 'госдума', 'суд', 'депутат', 'суда']",
 "['forbes_young', 'forbes', 'forbes_education', 'рассказываем', 'бизнес', 'бизнесменов', 'дайджест', 'фи

In [173]:
topic_model.visualize_topics()

In [67]:
import json

# Проверяем содержимое перед парсингом
print(data['Representative_Docs'].iloc[0])

# Исправляем ошибку парсинга JSON - возможно, строка требует предварительной обработки
try:
    # Пробуем очистить строку от лишних символов и заменить одинарные кавычки на двойные
    cleaned_json = data['Representative_Docs'].iloc[0].replace("'", '"')
    parsed_data = json.loads(cleaned_json)
    print("Успешно распарсили JSON")
    print(parsed_data)
except json.JSONDecodeError as e:
    print(f"Ошибка парсинга JSON: {e}")
    # Альтернативный подход - использовать ast.literal_eval для парсинга Python литералов
    import ast
    try:
        parsed_data = ast.literal_eval(data['Representative_Docs'].iloc[0])
        print("Успешно распарсили с помощью ast.literal_eval")
        print(parsed_data)
    except:
        print("Не удалось распарсить данные")

['Глава МЧС России Евгений Зиничев поддержал назначение своего бывшего заместителя Игоря Кобзева на должность временного исполняющего обязанности губернатора Иркутской области. Об этом сообщает ТАСС. По словам руководителя ведомства, Кобзев в новой должности будет внимательно относиться к проблемам каждого человека и справится с поставленными задачами и выразил надежду, что новый глава региона будет руководствоваться принципами неравнодушного подхода к проблемам каждого жителя Иркутской области. Зиничев рассказал, что проработал с ним полтора года, и за это время Кобзев показал себя как инициативный и грамотный сотрудник. «Под его руководством и при его личном участии была скорректирована модель риск-ориентированного подхода к объектам с массовым пребыванием людей, определены категории риска объектов», — поделился глава МЧС. Назначение также поддержал заместитель председателя правительства России Алексей Гордеев. Он охарактеризовал Кобзева как эффективного и системного руководителя, на

In [23]:
parsed_data

['Глава МВД Украины Арсен Аваков заявил, что Киев и Москва могут прийти к компромиссу в вопросе передачи Украине контроля над границей в Донбассе. Интервью с министром опубликовано на странице издания «Громадське» в Twitter. По его словам, в первое время контроль над границей могут осуществлять не погранвойска, а украинская полиция вместе с «представителями территориальных общин». При этом он указал, что это станет возможным только после того, как вооруженные формирования покинут территории самопровозглашенных республик. Аваков считает, что такой «переходный период» может продолжаться вплоть до года, но Украина «готова это пройти». При этом он указал, что участники «нормандского саммита» не дали согласия на такой вариант, а президент России Владимир Путин «не готов вернуть границу». Говоря о возможном компромиссе, Аваков отметил, что Киев может получить контроль над границей «не за месяц до местных выборов, а за два дня». Ранее президент Украины Владимир Зеленский заявил о необходимост

In [ ]:
topic_model.get_topic_info()["CustomName"]

In [56]:
# Визуализация кластеров новостей в 2D пространстве с помощью plotly
import plotly.express as px
import pandas as pd
import numpy as np

# Получаем координаты документов в 2D пространстве
embeddings_2d = zipped_embs

# Получаем данные о документах
doc_info = topic_model.get_document_info(news['text'].tolist())

# Создаем DataFrame для визуализации
plot_df = pd.DataFrame({
    'x': embeddings_2d.embedding_x,
    'y': embeddings_2d.embedding_y,
    'topic': embeddings_2d.topic,
    'text': doc_info.Document,
    'topic_name': doc_info.Name
})

# Создаем цветовую схему
colors = px.colors.qualitative.Plotly

# Создаем интерактивную визуализацию
fig = px.scatter(
    plot_df, 
    x='x', 
    y='y', 
    color='topic_name',
    hover_data=['text'],
    title='Кластеризация новостей по темам',
    color_discrete_sequence=colors,
    opacity=0.7,
    size_max=10
)

# Настраиваем внешний вид графика
fig.update_traces(marker=dict(size=8, line=dict(width=1, color='DarkSlateGrey')))
fig.update_layout(
    legend_title_text='Темы',
    xaxis_title="",
    yaxis_title="",
    xaxis=dict(showticklabels=False),
    yaxis=dict(showticklabels=False),
    plot_bgcolor='white'
)

# Отображаем график
fig.show()



NameError: name 'df' is not defined

In [44]:
topic_model.visualize_barchart(top_n_topics=30, n_words=10, title='Топ слов по темам')